In [ ]:
# --- Cell 1: environment + scratch space discovery -------------------------
import os, sys, platform, subprocess

print("project:", "OmniGen-v1")
print("python:", sys.version.split()[0])
print("platform:", platform.platform())

for p in ("/kaggle/temp", "/kaggle/working"):
    os.makedirs(p, exist_ok=True)
    st = os.statvfs(p)
    free = st.f_bavail * st.f_frsize / 1e9
    print(f"{p}: free={free:.1f} GB writable={os.access(p, os.W_OK)}")

print("--- GPU ---")
print(subprocess.run(["nvidia-smi",
                      "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "no nvidia-smi")

print("--- internet ---")
for u in ("https://huggingface.co", "https://github.com"):
    r = subprocess.run(["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}",
                        "--max-time", "30", u], capture_output=True, text=True)
    print(u, "->", r.stdout)


In [ ]:
# --- Cell 2: install dependencies for OmniGen inference -------------------
import subprocess, sys, os

pkgs = [
    "accelerate>=0.33.0",
    "diffusers>=0.30.0",
    "transformers>=4.45.0",
    "sentencepiece",
    "timm",
    "peft>=0.9.0",
    "datasets",
    "safetensors",
]
print("installing dependencies:", " ".join(pkgs), flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

print("installing OmniGen (using --no-deps to avoid PyTorch downgrade conflicts)...", flush=True)
installed = False
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-q", "git+https://github.com/VectorSpaceLab/OmniGen.git"], check=True)
    installed = True
except Exception as e:
    print("pip git install failed, trying fallback:", e, flush=True)

if not installed:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-q", "OmniGen>=1.0.3"], check=True)
        installed = True
    except Exception as e:
        print("pip pypi install failed:", e, flush=True)

if not installed or not os.path.exists("/kaggle/temp/OmniGen"):
    print("cloning OmniGen repository directly as safeguard...", flush=True)
    subprocess.run(["git", "clone", "--depth=1", "https://github.com/VectorSpaceLab/OmniGen.git", "/kaggle/temp/OmniGen"], check=False)

print("OmniGen setup completed.", flush=True)


In [ ]:
# --- Cell 3: load OmniGen pipeline + start server + open tunnels ----------
import os, re, subprocess, time, urllib.request, json, threading, http.server, sys
import base64, io, gc
import torch
from PIL import Image

if os.path.exists("/kaggle/temp/OmniGen") and "/kaggle/temp/OmniGen" not in sys.path:
    sys.path.insert(0, "/kaggle/temp/OmniGen")

from OmniGen import OmniGenPipeline

PORT = int("8080")
CTL_PORT = int("8081")
DEFAULT_STEPS = int("30" or "30")
DEFAULT_WIDTH = int("1024" or "1024")
DEFAULT_HEIGHT = int("1024" or "1024")
MODEL_REPO = "Shitao/OmniGen-v1"

STOP_REQUESTED = threading.Event()
GEN_LOCK = threading.Lock()

WORK = "/kaggle/temp/run"
UPLOAD_DIR = os.path.join(WORK, "uploads")
os.makedirs(WORK, exist_ok=True)
os.makedirs(UPLOAD_DIR, exist_ok=True)
CF_LOG = os.path.join(WORK, "cloudflared.log")
CTL_LOG = os.path.join(WORK, "cloudflared-ctl.log")

# ---- cloudflared binary ----
CF = "/kaggle/temp/cloudflared"
if not os.path.exists(CF):
    subprocess.run(["curl", "-sL", "-o", CF,
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                   check=True)
    os.chmod(CF, 0o755)

# ---- load pipeline ----
print(f"Loading OmniGen from {MODEL_REPO}...", flush=True)
pipe = OmniGenPipeline.from_pretrained(MODEL_REPO)
print("OmniGen pipeline loaded and ready on GPU!", flush=True)

# ---- Control Server on CTL_PORT (port 8081) ----
STOP_TOKEN = os.environ.get("CLUSTER_STOP_TOKEN") or os.urandom(8).hex()

class CtlHandler(http.server.BaseHTTPRequestHandler):
    def log_message(self, *a): pass
    def _send(self, code, payload):
        body = json.dumps(payload).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path.rstrip("/") in ("", "/health"):
            self._send(200, {"ok": True, "status": "up",
                             "stopping": STOP_REQUESTED.is_set(),
                             "model": "omnigen"})
        elif self.path.startswith("/stop/" + STOP_TOKEN):
            STOP_REQUESTED.set()
            self._send(200, {"ok": True, "message": "shutdown requested"})
            print("STOP_REQUESTED received via control tunnel", flush=True)
        else:
            self._send(404, {"ok": False, "error": "not found"})

    def do_POST(self):
        self.do_GET()

ctl = http.server.ThreadingHTTPServer(("127.0.0.1", CTL_PORT), CtlHandler)
threading.Thread(target=ctl.serve_forever, daemon=True).start()

# ---- OmniGen Image Server on PORT (port 8080) ----
HTML_PAGE = """<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>OmniGen - Multi-Image & Thumbnail Studio</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; }
  body { background: #0b0f19; color: #e1e4ea; min-height: 100vh; display: flex; flex-direction: column; align-items: center; padding: 24px 16px; }
  .card { background: #151b28; border: 1px solid #232d42; border-radius: 14px; width: 100%; max-width: 900px; padding: 28px; box-shadow: 0 12px 32px rgba(0,0,0,0.5); margin-bottom: 24px; }
  h1 { font-size: 1.7rem; color: #fff; margin-bottom: 6px; display: flex; align-items: center; gap: 10px; }
  .badge { font-size: 0.75rem; background: #2563eb; color: #fff; padding: 3px 10px; border-radius: 6px; font-weight: 700; }
  p.subtitle { color: #94a3b8; font-size: 0.92rem; margin-bottom: 20px; line-height: 1.4; }
  .notice { background: #1e293b; border-left: 4px solid #38bdf8; padding: 12px 16px; border-radius: 6px; margin-bottom: 20px; font-size: 0.88rem; color: #cbd5e1; }
  .notice code { color: #38bdf8; font-weight: 600; }
  label { display: block; font-size: 0.85rem; font-weight: 600; color: #cbd5e1; margin-bottom: 6px; }
  textarea { width: 100%; background: #0b0f19; border: 1px solid #334155; border-radius: 8px; color: #fff; padding: 12px; font-size: 0.95rem; min-height: 100px; resize: vertical; margin-bottom: 20px; }
  textarea:focus { outline: none; border-color: #38bdf8; }
  .upload-grid { display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 14px; margin-bottom: 20px; }
  .upload-box { background: #0f172a; border: 2px dashed #334155; border-radius: 10px; padding: 12px; text-align: center; cursor: pointer; transition: border-color 0.2s; position: relative; }
  .upload-box:hover { border-color: #38bdf8; }
  .upload-box input { display: none; }
  .upload-box img { max-width: 100%; max-height: 120px; border-radius: 6px; display: none; margin-top: 8px; }
  .upload-tag { font-size: 0.8rem; font-weight: 700; color: #38bdf8; margin-bottom: 4px; }
  .upload-label { font-size: 0.75rem; color: #64748b; }
  .row { display: grid; grid-template-columns: 1fr 1fr 1fr 1fr; gap: 12px; margin-bottom: 20px; }
  select, input { width: 100%; background: #0b0f19; border: 1px solid #334155; border-radius: 8px; color: #fff; padding: 10px; font-size: 0.9rem; }
  button { width: 100%; background: #0284c7; color: #fff; border: none; padding: 14px 20px; font-size: 1.05rem; font-weight: 700; border-radius: 8px; cursor: pointer; transition: background 0.2s; }
  button:hover { background: #0369a1; }
  button:disabled { background: #475569; cursor: not-allowed; }
  #status { margin-top: 16px; font-size: 0.95rem; color: #38bdf8; text-align: center; }
  #preview { margin-top: 24px; text-align: center; }
  #preview img { max-width: 100%; border-radius: 10px; box-shadow: 0 6px 20px rgba(0,0,0,0.6); }
  .download-btn { margin-top: 14px; display: inline-block; background: #10b981; color: #fff; text-decoration: none; padding: 10px 20px; border-radius: 6px; font-size: 0.95rem; font-weight: 700; }
</style>
</head>
<body>
<div class="card">
  <h1>OmniGen <span class="badge">MULTI-IMAGE STUDIO</span></h1>
  <p class="subtitle">Unified Multi-Image Composition & Thumbnail Generator running on Kaggle GPU</p>

  <div class="notice">
    <strong>Thumbnail Multi-Subject Tip:</strong> Upload your reference photos below. In your prompt, reference them with <code>&lt;img&gt;&lt;|image_1|&gt;&lt;/img&gt;</code>, <code>&lt;img&gt;&lt;|image_2|&gt;&lt;/img&gt;</code>, etc. You can also leave the upload boxes empty for pure text-to-image generation!
  </div>

  <div class="upload-grid">
    <div class="upload-box" onclick="document.getElementById('file1').click()">
      <div class="upload-tag">&lt;|image_1|&gt;</div>
      <div class="upload-label" id="lbl1">Click to upload Subject 1</div>
      <input type="file" id="file1" accept="image/*" onchange="previewImg(1)">
      <img id="prev1" alt="Image 1">
    </div>
    <div class="upload-box" onclick="document.getElementById('file2').click()">
      <div class="upload-tag">&lt;|image_2|&gt;</div>
      <div class="upload-label" id="lbl2">Click to upload Subject 2</div>
      <input type="file" id="file2" accept="image/*" onchange="previewImg(2)">
      <img id="prev2" alt="Image 2">
    </div>
    <div class="upload-box" onclick="document.getElementById('file3').click()">
      <div class="upload-tag">&lt;|image_3|&gt;</div>
      <div class="upload-label" id="lbl3">Click to upload Subject 3</div>
      <input type="file" id="file3" accept="image/*" onchange="previewImg(3)">
      <img id="prev3" alt="Image 3">
    </div>
  </div>

  <label for="prompt">Prompt Instruction</label>
  <textarea id="prompt" placeholder="e.g. A dramatic YouTube thumbnail of <img><|image_1|></img> and <img><|image_2|></img> facing each other across a geopolitical map, intense cinematic lighting, photorealistic 8k"></textarea>

  <div class="row">
    <div>
      <label for="size">Resolution</label>
      <select id="size">
        <option value="1024x1024" selected>1024 × 1024</option>
        <option value="1024x768">1024 × 768 (Landscape)</option>
        <option value="768x1024">768 × 1024 (Portrait)</option>
        <option value="768x768">768 × 768</option>
        <option value="512x512">512 × 512 (Fastest)</option>
      </select>
    </div>
    <div>
      <label for="steps">Steps</label>
      <select id="steps">
        <option value="25">25 (Fast)</option>
        <option value="30" selected>30 (Balanced)</option>
        <option value="40">40 (Quality)</option>
        <option value="50">50 (Max Detail)</option>
      </select>
    </div>
    <div>
      <label for="cfg">Guidance Scale</label>
      <input id="cfg" type="number" step="0.5" value="3.0">
    </div>
    <div>
      <label for="seed">Seed (optional)</label>
      <input id="seed" type="number" placeholder="Random">
    </div>
  </div>

  <button id="genBtn" onclick="generate()">Generate Image</button>
  <div id="status"></div>
  <div id="preview"></div>
</div>

<script>
const imgs = [null, null, null];

function previewImg(idx) {
  const fileInput = document.getElementById('file' + idx);
  if (!fileInput.files || !fileInput.files[0]) return;
  const reader = new FileReader();
  reader.onload = function(e) {
    imgs[idx - 1] = e.target.result;
    const preview = document.getElementById('prev' + idx);
    preview.src = e.target.result;
    preview.style.display = 'block';
    document.getElementById('lbl' + idx).textContent = fileInput.files[0].name.slice(0, 18);
  };
  reader.readAsDataURL(fileInput.files[0]);
}

async function generate() {
  const prompt = document.getElementById('prompt').value.trim();
  if (!prompt) return alert('Please enter a prompt');
  const size = document.getElementById('size').value;
  const steps = parseInt(document.getElementById('steps').value);
  const guidance_scale = parseFloat(document.getElementById('cfg').value) || 3.0;
  const seedVal = document.getElementById('seed').value;
  const btn = document.getElementById('genBtn');
  const status = document.getElementById('status');
  const preview = document.getElementById('preview');

  btn.disabled = true;
  preview.innerHTML = '';
  const startTime = Date.now();
  status.textContent = 'Generating on Kaggle GPU... (~20-40s)';

  try {
    const input_images = imgs.filter(Boolean);
    const payload = { prompt, size, steps, guidance_scale, input_images, response_format: 'b64_json' };
    if (seedVal) payload.seed = parseInt(seedVal);

    const resp = await fetch('/v1/images/generations', {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify(payload)
    });
    if (!resp.ok) {
      const err = await resp.text();
      throw new Error(`Server returned HTTP ${resp.status}: ${err}`);
    }
    const data = await resp.json();
    const b64 = data.data[0].b64_json;
    const elapsed = ((Date.now() - startTime) / 1000).toFixed(1);
    status.textContent = `Completed in ${elapsed}s!`;
    const imgSrc = 'data:image/png;base64,' + b64;
    preview.innerHTML = `
      <img src="${imgSrc}" alt="Generated Image"><br>
      <a class="download-btn" href="${imgSrc}" download="omnigen-${Date.now()}.png">Download Full Resolution</a>
    `;
  } catch (e) {
    status.textContent = 'Error: ' + e.message;
  } finally {
    btn.disabled = false;
  }
}
</script>
</body>
</html>
"""

AUTH_USER = "alamgir-zk" or os.environ.get("CLUSTER_AUTH_USER", "")
AUTH_PASS = "Zaid2002@kaggle" or os.environ.get("CLUSTER_AUTH_PASS", "")
AUTH_KEY = "sk-cluster-ai883742-secret-key" or os.environ.get("CLUSTER_API_KEY", "")
AUTH_ENABLED = bool((AUTH_USER and AUTH_PASS) or AUTH_KEY)

class ServerHandler(http.server.BaseHTTPRequestHandler):
    def log_message(self, *a): pass

    def _check_auth(self):
        if not AUTH_ENABLED:
            return True
        path = self.path.split("?")[0].rstrip("/")
        if path in ("/health", "/v1/health", "/healthz"):
            return True
        auth = self.headers.get("Authorization", "")
        if auth.startswith("Basic "):
            try:
                creds = base64.b64decode(auth[6:].strip()).decode("utf-8")
                u, p = creds.split(":", 1)
                if u == AUTH_USER and p == AUTH_PASS:
                    return True
            except Exception:
                pass
        elif auth.startswith("Bearer "):
            token = auth[7:].strip()
            if (AUTH_KEY and token == AUTH_KEY) or (AUTH_PASS and token == AUTH_PASS):
                return True
            try:
                creds = base64.b64decode(token).decode("utf-8")
                u, p = creds.split(":", 1)
                if u == AUTH_USER and p == AUTH_PASS:
                    return True
            except Exception:
                pass
        self.send_response(401)
        self.send_header("WWW-Authenticate", 'Basic realm="AI Cluster Protected"')
        self.send_header("Content-Type", "application/json")
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        self.wfile.write(b'{"error": "Unauthorized: valid username and password required"}\n')
        return False

    def _send_json(self, code, payload):
        body = json.dumps(payload).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        self.wfile.write(body)

    def do_OPTIONS(self):
        self.send_response(200)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Content-Type, Authorization")
        self.end_headers()

    def do_HEAD(self):
        if not self._check_auth(): return
        self.do_GET()

    def do_GET(self):
        if not self._check_auth(): return
        path = self.path.split("?")[0].rstrip("/")
        if path in ("", "/index.html"):
            body = HTML_PAGE.encode("utf-8")
            self.send_response(200)
            self.send_header("Content-Type", "text/html; charset=utf-8")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)
        elif path in ("/health", "/v1/health"):
            self._send_json(200, {
                "status": "ok",
                "model": "omnigen",
                "architecture": "omnigen",
                "device": "cuda" if torch.cuda.is_available() else "cpu"
            })
        elif path in ("/models", "/v1/models"):
            self._send_json(200, {
                "data": [{
                    "id": "omnigen",
                    "object": "model",
                    "type": "image-generation"
                }]
            })
        else:
            self._send_json(404, {"error": f"Path {path} not found"})

    def do_POST(self):
        if not self._check_auth(): return
        path = self.path.split("?")[0].rstrip("/")
        content_len = int(self.headers.get("Content-Length", 0))
        post_data = self.rfile.read(content_len) if content_len > 0 else b"{}"
        try:
            req = json.loads(post_data.decode("utf-8"))
        except Exception:
            return self._send_json(400, {"error": "Invalid JSON payload"})

        if path in ("/v1/images/generations", "/generate"):
            prompt = req.get("prompt", "")
            if not prompt:
                return self._send_json(400, {"error": "Missing prompt"})

            size = req.get("size", "1024x1024")
            try:
                width, height = map(int, size.lower().split("x"))
            except Exception:
                width = int(req.get("width", DEFAULT_WIDTH))
                height = int(req.get("height", DEFAULT_HEIGHT))

            steps = int(req.get("steps", DEFAULT_STEPS))
            guidance_scale = float(req.get("guidance_scale", 3.0))
            img_guidance_scale = float(req.get("img_guidance_scale", 1.6))
            seed = int(req.get("seed", 0)) if req.get("seed") is not None else None

            # Process input images if provided
            raw_input_images = req.get("input_images", [])
            local_paths = []
            for i, item in enumerate(raw_input_images):
                if not item: continue
                dest_path = os.path.join(UPLOAD_DIR, f"input_{i}.png")
                if item.startswith("data:"):
                    # base64 data URI
                    header, b64 = item.split(",", 1)
                    with open(dest_path, "wb") as f:
                        f.write(base64.b64decode(b64))
                    local_paths.append(dest_path)
                elif item.startswith("http://") or item.startswith("https://"):
                    req_img = urllib.request.Request(item, headers={"User-Agent": "curl/8.5.0"})
                    with urllib.request.urlopen(req_img) as r, open(dest_path, "wb") as f:
                        f.write(r.read())
                    local_paths.append(dest_path)
                elif os.path.exists(item):
                    local_paths.append(item)

            print(f"Generating OmniGen ({width}x{height}, {steps} steps, {len(local_paths)} ref images)...", flush=True)
            t0 = time.time()
            with GEN_LOCK:
                out = pipe(
                    prompt=prompt,
                    input_images=local_paths if local_paths else None,
                    height=height,
                    width=width,
                    num_inference_steps=steps,
                    guidance_scale=guidance_scale,
                    img_guidance_scale=img_guidance_scale,
                    seed=seed,
                )
                img = out[0]
            elapsed = round(time.time() - t0, 2)
            print(f"Generation done in {elapsed}s", flush=True)

            buf = io.BytesIO()
            img.save(buf, format="PNG")
            b64_bytes = base64.b64encode(buf.getvalue()).decode("utf-8")

            if path == "/generate":
                return self._send_json(200, {
                    "image_b64": b64_bytes,
                    "prompt": prompt,
                    "width": width,
                    "height": height,
                    "steps": steps,
                    "elapsed_seconds": elapsed
                })
            else:
                return self._send_json(200, {
                    "created": int(time.time()),
                    "data": [
                        {
                            "b64_json": b64_bytes,
                            "revised_prompt": prompt
                        }
                    ]
                })
        else:
            self._send_json(404, {"error": f"POST {path} not supported"})

server = http.server.ThreadingHTTPServer(("127.0.0.1", PORT), ServerHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
print("OmniGen server running on port", PORT, flush=True)

# ---- publish both tunnels ----
CF_TOKEN = "eyJhIjoiYzBiNGZhMjIzYWRkOTNhZDAxOTk4YmM0MDhmNzhjZDMiLCJzIjoiNk42Sm1KZERGdUFrUDVHcUpjekdvcnVHbHV0MFdVL0ZPdDFmN3k0SW4rVT0iLCJ0IjoiYmM2ZjA4OGQtYWU4ZC00ZmU3LWEzMmQtMjdjYTA3YWFiZTkwIn0=" or os.environ.get("CLOUDFLARE_TUNNEL_TOKEN", "")
STATIC_URL = "https://ai.883742.xyz" or os.environ.get("CLUSTER_URL", "")

def start_public_tunnel(url, log_path):
    fh = open(log_path, "w")
    if CF_TOKEN:
        print("Starting named Cloudflare tunnel with token...", flush=True)
        proc = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "run", "--token", CF_TOKEN, "--url", url],
                                stdout=fh, stderr=subprocess.STDOUT)
        time.sleep(5)
        if proc.poll() is not None:
            fh.flush()
            print("cloudflared failed to start! Log tail:", flush=True)
            print(open(log_path).read()[-2000:], flush=True)
            raise RuntimeError("named cloudflared tunnel exited immediately")
        return STATIC_URL or "https://ai.883742.xyz"
    else:
        subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", url],
                         stdout=fh, stderr=subprocess.STDOUT)
        for _ in range(60):
            time.sleep(2)
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log_path).read())
            if m:
                return m.group(0)
        return None

def start_tunnel(url, log_path):
    fh = open(log_path, "w")
    subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", url],
                     stdout=fh, stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(2)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log_path).read())
        if m:
            return m.group(0)
    return None

public_url = start_public_tunnel(f"http://127.0.0.1:{PORT}", CF_LOG)
control_url = start_tunnel(f"http://127.0.0.1:{CTL_PORT}", CTL_LOG)

if not public_url:
    print("cloudflared log:", open(CF_LOG).read()[-2000:], flush=True)
    raise RuntimeError("could not obtain a public tunnel URL")
if not control_url:
    print("control cloudflared log:", open(CTL_LOG).read()[-2000:], flush=True)
    raise RuntimeError("could not obtain a control tunnel URL")

with open("/kaggle/working/CONTROL_URL.txt", "w") as f:
    f.write(public_url + "\n")
with open("/kaggle/working/STOP_URL.txt", "w") as f:
    f.write(control_url + "/stop/" + STOP_TOKEN + "\n")

print("=" * 72)
print("CONTROL_URL=" + public_url)
print("STOP_URL=" + control_url + "/stop/" + STOP_TOKEN)
print("MODEL=" + "Shitao/OmniGen-v1")
print("=" * 72)
print(f"Web UI         : {public_url}")
print(f"API endpoint   : {public_url}/v1/images/generations")
print(f"Health         : {public_url}/health")
print("Stop from PC   : ctr.sh stop   (uses the control tunnel above)")


In [ ]:
# --- Cell 5: keep the tunnel alive (this is what you keep running) ---------
import time, urllib.request

PORT = int("8080")
SERVE_MINUTES = int("660")
AUTH_USER = "alamgir-zk"
AUTH_PASS = "Zaid2002@kaggle"

url = open("/kaggle/working/CONTROL_URL.txt").read().strip()
print(f"serving for ~{SERVE_MINUTES} minutes at {url}", flush=True)

for minute in range(SERVE_MINUTES):
    # Wait up to 60s, but poll the stop flag every 5s so shutdown is prompt.
    for _ in range(12):
        if STOP_REQUESTED.is_set():
            break
        time.sleep(5)
    if STOP_REQUESTED.is_set():
        print("STOP received from control tunnel -> ending session", flush=True)
        break
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/health")
        if AUTH_USER and AUTH_PASS:
            import base64
            auth_str = base64.b64encode(f"{AUTH_USER}:{AUTH_PASS}".encode()).decode()
            req.add_header("Authorization", f"Basic {auth_str}")
        with urllib.request.urlopen(req, timeout=5) as r:
            status = str(r.status)
    except Exception as e:
        status = f"err {e}"
    if minute % 5 == 0:
        # Re-print periodically so the URL can always be recovered from logs.
        print(f"[{minute}m] health={status} CONTROL_URL={url}", flush=True)

print("SERVE_WINDOW_COMPLETE", flush=True)
